<a href="https://colab.research.google.com/github/tala-aleid/tala-aleid-DataScience-GenAI-Submissions/blob/main/Assignment6/6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [1]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 32120.334
Epoch [10/100], Loss: 29374.1445
Epoch [10/100], Loss: 25010.5703
Epoch [10/100], Loss: 29507.3301
Epoch [10/100], Loss: 34808.7852
Epoch [10/100], Loss: 27592.7598
Epoch [10/100], Loss: 28297.0918
Epoch [10/100], Loss: 50481.4336
Epoch [20/100], Loss: 29007.0293
Epoch [20/100], Loss: 29015.5801
Epoch [20/100], Loss: 30229.7734
Epoch [20/100], Loss: 33410.0508
Epoch [20/100], Loss: 30414.4473
Epoch [20/100], Loss: 25421.7363
Epoch [20/100], Loss: 30926.0645
Epoch [20/100], Loss: 14945.4023
Epoch [30/100], Loss: 27828.0137
Epoch [30/100], Loss: 24514.5547
Epoch [30/100], Loss: 25953.5703
Epoch [30/100], Loss: 32184.1094
Epoch [30/100], Loss: 41761.7109
Epoch [30/100], Loss: 30872.2695
Epoch [30/100], Loss: 23678.3047
Epoch [30/100], Loss: 34152.4062
Epoch [40/100], Loss: 32650.4961
Epoch [40/100], Loss: 31079.0918
Epoch [40/100], Loss: 29771.2793
Epoch [40/100], Loss: 27921.9844
Epoch [40/100], Loss: 20065.9902
Epoch [40/100], Loss: 29324.2188
Epoch [40/1

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 23914.3251953125


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,8.536978,219.0
1,8.037097,70.0
2,8.366678,202.0
3,9.895477,230.0
4,8.050220,111.0
...,...,...
84,7.437809,153.0
85,6.927489,98.0
86,6.448459,37.0
87,6.609137,63.0


### Exercise 1: Training for 1,000 Epochs

In [11]:
# Re-initialize the model and optimizer to train from scratch
model_ex1 = DiabetesModel().to(device)
criterion_ex1 = nn.MSELoss()
optimiser_ex1 = optim.Adam(model_ex1.parameters(), lr=0.001)

# Set epochs to 1000
epochs_ex1 = 1000

print("Training the model for 1,000 epochs...")
for epoch in range(epochs_ex1):
    model_ex1.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimiser_ex1.zero_grad()
        outputs = model_ex1(inputs)
        loss = criterion_ex1(outputs, targets)
        loss.backward()
        optimiser_ex1.step()

    # Print loss every 100 epochs for cleaner output
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs_ex1}], Loss: {round(loss.item(), 4)}')

Training the model for 1,000 epochs...
Epoch [100/1000], Loss: 25780.1758
Epoch [200/1000], Loss: 4543.1436
Epoch [300/1000], Loss: 3808.8555
Epoch [400/1000], Loss: 2121.9746
Epoch [500/1000], Loss: 5399.3662
Epoch [600/1000], Loss: 2326.48
Epoch [700/1000], Loss: 10912.7725
Epoch [800/1000], Loss: 829.5256
Epoch [900/1000], Loss: 1005.8207
Epoch [1000/1000], Loss: 822.7748


Now let's evaluate our updated model on the test dataset and compute the new Average MSE.

In [12]:
model_ex1.eval()
mse_values_ex1 = []
predictions_ex1 = []
actuals_ex1 = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model_ex1(inputs)

        # Track loss
        mse = criterion_ex1(outputs, targets)
        mse_values_ex1.append(mse.item())

        # Track outputs for comparison
        predictions_ex1.extend(outputs.cpu().numpy())
        actuals_ex1.extend(targets.cpu().numpy())

avg_mse_ex1 = np.mean(mse_values_ex1)
print(f"New Average MSE on test set (1,000 epochs): {avg_mse_ex1}")

New Average MSE on test set (1,000 epochs): 2864.1314697265625


Let's create a DataFrame to visually compare the new predictions side-by-side with the actual values.

In [13]:
results_df_ex1 = pd.DataFrame({
    'Predicted': np.array(predictions_ex1).flatten(),
    'Actual': np.array(actuals_ex1).flatten()
})
display(results_df_ex1.head(20))

,Predicted,Actual
0,146.337723,219.0
1,174.073288,70.0
2,142.235550,202.0
3,291.551575,230.0
4,127.937843,111.0
5,101.676514,84.0
6,239.018265,242.0
7,191.223770,272.0
8,92.720528,94.0
9,113.419472,96.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?